In [1]:
from pathlib import Path
from typing import List, Tuple, Dict
import json, random, itertools, os, gc, statistics as st
import seqeval

import numpy as np
import pandas as pd
from datasets import Dataset, DatasetDict, concatenate_datasets, load_dataset
from sklearn.model_selection import KFold, train_test_split
from collections import Counter
from sentence_transformers import SentenceTransformer
from sklearn.neighbors import BallTree
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity, pairwise_distances
from sklearn.cluster import KMeans

import torch
from transformers import (
    AutoTokenizer,
        AutoModelForTokenClassification,
    TrainingArguments,
    Trainer,
    DataCollatorForTokenClassification
)

from seqeval.metrics import f1_score, classification_report
from evaluate import load as load_metric
from tqdm.auto import tqdm

c:\Users\user\miniconda3\envs\lora-gpu\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
from numpy.linalg import norm

# Configuração e Verificação Inicial

In [3]:
EXPORT_DIR = "lener"  # mude o nome se quiser
Path(EXPORT_DIR).mkdir(parents=True, exist_ok=True)
os.chdir(EXPORT_DIR)

In [4]:
SEED_GLOBAL = 42
random.seed(SEED_GLOBAL)
np.random.seed(SEED_GLOBAL)
torch.manual_seed(SEED_GLOBAL)

MODEL_NAME = "neuralmind/bert-base-portuguese-cased"

In [5]:
lener_ds = load_dataset("peluz/lener_br", trust_remote_code = True)

In [6]:
lener_ds

DatasetDict({
    train: Dataset({
        features: ['id', 'tokens', 'ner_tags'],
        num_rows: 7828
    })
    validation: Dataset({
        features: ['id', 'tokens', 'ner_tags'],
        num_rows: 1177
    })
    test: Dataset({
        features: ['id', 'tokens', 'ner_tags'],
        num_rows: 1390
    })
})

In [7]:
lener_full = concatenate_datasets(
    [lener_ds["train"], lener_ds["validation"], lener_ds["test"]]
)

In [8]:
tags = [
    "O",
    "B-ORGANIZACAO",
    "I-ORGANIZACAO",
    "B-PESSOA",
    "I-PESSOA",
    "B-TEMPO",
    "I-TEMPO",
    "B-LOCAL",
    "I-LOCAL",
    "B-LEGISLACAO",
    "I-LEGISLACAO",
    "B-JURISPRUDENCIA",
    "I-JURISPRUDENCIA",
]

In [9]:
label2id = {l: i for i, l in enumerate(tags)}
id2label = {i: l for i, l in enumerate(tags)}
NUM_LABELS = len(tags)

In [10]:
label2id

{'O': 0,
 'B-ORGANIZACAO': 1,
 'I-ORGANIZACAO': 2,
 'B-PESSOA': 3,
 'I-PESSOA': 4,
 'B-TEMPO': 5,
 'I-TEMPO': 6,
 'B-LOCAL': 7,
 'I-LOCAL': 8,
 'B-LEGISLACAO': 9,
 'I-LEGISLACAO': 10,
 'B-JURISPRUDENCIA': 11,
 'I-JURISPRUDENCIA': 12}

In [11]:
def decode_labels(example):
    example["ner_tags_str"] = [tags[i] for i in example["ner_tags"]]
    return example


lener_full = lener_full.map(decode_labels)

In [12]:
lener_full

Dataset({
    features: ['id', 'tokens', 'ner_tags', 'ner_tags_str'],
    num_rows: 10395
})

In [13]:
NUM_LABELS

13

# Splits

In [14]:
def split_standard(ds: Dataset) -> DatasetDict:
    """Usa coluna trainingTest do CSV (80/20 original)."""
    if "trainingTest" not in df.columns:
        raise ValueError("CSV não contém a coluna 'trainingTest'")
    train_ids = df.loc[df["trainingTest"] == "training", SENT_COL].unique()
    test_ids = df.loc[df["trainingTest"] == "test", SENT_COL].unique()
    return DatasetDict(
        train=ds.filter(lambda ex: ex["sentence_id"] in train_ids),
        dev=ds.filter(lambda ex: ex["sentence_id"] in test_ids),
    )

In [15]:
def random_splits(
    ds: Dataset, test_size=0.2, seeds: List[int] = range(30)
) -> List[DatasetDict]:
    triples = []
    for s in seeds:
        train, dev = ds.train_test_split(test_size=test_size, seed=s).values()
        triples.append(DatasetDict(train=train, dev=dev))
    return triples

In [16]:
# def split_heur_length(ds: Dataset, top_pct: float = 0.20) -> DatasetDict:
#     lengths = np.array([len(t) for t in ds["tokens"]])
#     thr = np.percentile(lengths, 100 * (1 - top_pct))
#     mask = lengths >= thr
#     return DatasetDict(train=ds.filter(~mask), dev=ds.filter(mask))


def split_heur_length(ds: Dataset, top_pct: float = 0.20) -> DatasetDict:
    """20 % das sentenças mais longas viram conjunto de validação (dev)."""
    lengths = np.array([len(t) for t in ds["tokens"]])
    thr = np.percentile(lengths, 100 * (1 - top_pct))  
    mask = lengths >= thr  

    dev_idx = np.where(mask)[0].tolist()  # índices → list[int]
    train_idx = np.where(~mask)[0].tolist()

    return DatasetDict(
        train=ds.select(train_idx),
        dev=ds.select(dev_idx),
    )

    # Tamanho da sentenças


# def split_heur_rare(ds: Dataset, freq_thr: int = 5) -> DatasetDict:
#     freq = Counter(w.lower() for sent in ds["tokens"] for w in sent)
#     rare = {w for w, c in freq.items() if c <= freq_thr}

#     def has_rare(example):
#         return any(w.lower() in rare for w in example["tokens"])

#     return DatasetDict(
#         train=ds.filter(lambda ex: not has_rare(ex)), dev=ds.filter(has_rare)
#     )


def split_heur_rare(ds: Dataset, freq_thr: int = 5) -> DatasetDict:
    freq = Counter(w.lower() for sent in ds["tokens"] for w in sent)
    rare = {w for w, c in freq.items() if c <= freq_thr}

    keep_dev = []
    for sent in ds["tokens"]:
        print(sent)
        keep_dev.append(any(w.lower() in rare for w in sent))

    dev_idx = [i for i, x in enumerate(keep_dev) if x]
    train_idx = [i for i, x in enumerate(keep_dev) if not x]

    return DatasetDict(
        train=ds.select(train_idx),
        dev=ds.select(dev_idx),
    )

    # Raridade dos tokens

In [17]:
def split_adversarial(ds: Dataset, pct_test: float = 0.20) -> DatasetDict:
    k = int(len(ds) * pct_test)
    model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
    embeds = model.encode([" ".join(t) for t in ds["tokens"]], show_progress_bar=False)
    tree = BallTree(embeds, leaf_size=40)

    idx_train, idx_test = set(range(len(ds))), []
    # semente = ponto mais central
    seed_idx = np.argmax(np.linalg.norm(embeds - embeds.mean(0), axis=1))
    idx_train.remove(seed_idx)
    idx_test.append(seed_idx)
    print(k)
    while len(idx_test) < k:
        print(len(idx_test))
        dists, _ = tree.query(embeds[list(idx_train)], k=1, return_distance=True)
        nxt = list(idx_train)[int(np.argmax(dists))]
        idx_train.remove(nxt)
        idx_test.append(nxt)

    return DatasetDict(
        train=ds.select(sorted(idx_train)),
        dev=ds.select(sorted(idx_test)),
    )

    # Maximizando Wassertein Distance


def split_adversarial_fast(ds: Dataset, pct_test: float = 0.20) -> DatasetDict:
    """
    Farthest-Point Sampling aproximando Wasserstein – versão vetorizada.
    Seleciona pct_test (~20 %) das sentenças como conjunto 'dev'.
    """
    k = int(len(ds) * pct_test)
    model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

    embeds = model.encode(
        [" ".join(t) for t in ds["tokens"]],
        show_progress_bar=True,
        convert_to_numpy=True,
        normalize_embeddings=True,  # acelera distância euclidiana ≈ cos
    )

    n = embeds.shape[0]
    idx_all = np.arange(n)

    # 1) ponto mais "central" (norma mais distante da média)
    seed_idx = np.argmax(np.linalg.norm(embeds - embeds.mean(0), axis=1))
    selected = [seed_idx]

    # 2) vetor de distâncias mínimas a qualquer ponto já escolhido
    min_dists = np.linalg.norm(embeds - embeds[seed_idx], axis=1)

    while len(selected) < k:
        next_idx = np.argmax(min_dists)
        selected.append(next_idx)

        # atualiza min_dists com a distância ao novo ponto — tudo de uma vez
        d_new = np.linalg.norm(embeds - embeds[next_idx], axis=1)
        min_dists = np.minimum(min_dists, d_new)

    train_idx = np.setdiff1d(idx_all, selected, assume_unique=True)

    return DatasetDict(
        train=ds.select(train_idx.tolist()),
        dev=ds.select(selected),
    )

In [18]:
# def loc_split(
#     dataset: Dataset, pct_test: float = 0.20, ngram: int = 4, seed: int = 42
# ) -> DatasetDict:
#     """
#     Split baseado em baixa sobreposição léxica (4-gram Jaccard).
#     Teste = pct_test das sentenças com menor overlap em relação ao pool.
#     """
#     # 1. Texto plano por sentença
#     docs = [" ".join(toks) for toks in dataset["tokens"]]

#     # 2. Vetorizar 4-grams (binário)
#     vect = CountVectorizer(
#         analyzer="word", ngram_range=(ngram, ngram), binary=True
#     ).fit(docs)
#     X = vect.transform(docs)

#     # 3. Similaridade Jaccard aproximada com matriz binária
#     # Jaccard(A,B) = |A∩B|/|A∪B| = 1 - |AΔB|/|A∪B|
#     # Usamos: overlap = (A·Bᵀ) / (|A|+|B|-A·Bᵀ)
#     bin_counts = X.sum(axis=1).A1

#     # Para cada doc i, escolhemos vizinho + próximo (fast):
#     from sklearn.metrics.pairwise import cosine_similarity

#     # (cosine no binário ∝ |A∩B|)
#     sim = cosine_similarity(X, dense_output=False)
#     # Soma dos top-k overlaps (k=5) como score
#     k = 5
#     topk = np.zeros(len(dataset))
#     for i in range(sim.shape[0]):
#         row = sim.getrow(i).toarray()[0]
#         idx = np.argpartition(-row, range(1, k + 1))[1 : k + 1]
#         # overlap ≈ |∩|
#         inter = row[idx] * bin_counts[i]
#         uni = bin_counts[i] + bin_counts[idx] - inter
#         topk[i] = (inter / uni).mean()

#     # 4. Ordenar por overlap crescente ⇒ mais “novos” vão p/ teste
#     order = np.argsort(topk)
#     n_test = int(len(dataset) * pct_test)
#     test_idx = order[:n_test]
#     train_idx = order[n_test:]

#     return DatasetDict(
#         {"train": dataset.select(train_idx), "test": dataset.select(test_idx)}
#     )

In [19]:
def loc_split(
    dataset: Dataset,
    pct_test: float = 0.20,
    pct_val: float = 0.10,
    ngram: int = 4,
    seed: int = 42,
) -> DatasetDict:
    """
    Divide por sobreposição léxica (n-gram Jaccard).
    Frações independentes para teste e validação.
    """
    docs = [" ".join(toks) for toks in dataset["tokens"]]

    vect = CountVectorizer(
        analyzer="word", ngram_range=(ngram, ngram), binary=True
    ).fit(docs)
    X = vect.transform(docs)
    bin_counts = X.sum(axis=1).A1

    sim = cosine_similarity(X, dense_output=False)
    k = 5
    topk = np.zeros(len(dataset))
    for i in range(sim.shape[0]):
        row = sim.getrow(i).toarray()[0]
        idx = np.argpartition(-row, range(1, k + 1))[1 : k + 1]
        inter = row[idx] * bin_counts[i]
        uni = bin_counts[i] + bin_counts[idx] - inter
        topk[i] = (inter / uni).mean()

    order = np.argsort(topk)  # baixo → alto overlap
    n_test = int(len(dataset) * pct_test)
    n_val = int(len(dataset) * pct_val)

    test_idx = order[:n_test]
    val_idx = order[n_test : n_test + n_val]
    train_idx = order[n_test + n_val :]

    return DatasetDict(
        {
            "train": dataset.select(train_idx),
            "val": dataset.select(val_idx),
            "test": dataset.select(test_idx),
        }
    )

In [20]:
def semantic_cluster_split(
    dataset: Dataset,
    pct_test: float = 0.20,
    pct_val: float = 0.10,
    k: int | None = None,
    seed: int = 42,
) -> DatasetDict:
    """
    Clusters SBERT → reserva clusters distantes para test/val.
    """
    if k is None:
        k = int(np.sqrt(len(dataset)))

    sbert = SentenceTransformer(
        "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
    )
    embeddings = sbert.encode(
        [" ".join(t) for t in tqdm(dataset["tokens"])],
        batch_size=64,
        show_progress_bar=False,
    )

    km = KMeans(n_clusters=k, random_state=seed, n_init=10).fit(embeddings)
    labels = km.labels_
    centroids = km.cluster_centers_

    global_center = embeddings.mean(0, keepdims=True)
    dists = pairwise_distances(centroids, global_center).flatten()

    clusters_sorted = np.argsort(-dists)  # mais distantes primeiro
    test_clusters, val_clusters = set(), set()
    total_test = total_val = 0
    n_test = int(len(dataset) * pct_test)
    n_val = int(len(dataset) * pct_val)

    for c in clusters_sorted:
        size = np.sum(labels == c)
        if total_test < n_test:  # preenche primeiro o teste
            test_clusters.add(c)
            total_test += size
        elif total_val < n_val:  # depois a validação
            val_clusters.add(c)
            total_val += size
        if total_test >= n_test and total_val >= n_val:
            break

    test_idx = np.where([lbl in test_clusters for lbl in labels])[0]
    val_idx = np.where([lbl in val_clusters for lbl in labels])[0]
    train_idx = np.where(
        [lbl not in test_clusters and lbl not in val_clusters for lbl in labels]
    )[0]

    return DatasetDict(
        {
            "train": dataset.select(train_idx),
            "val": dataset.select(val_idx),
            "test": dataset.select(test_idx),
        }
    )

In [21]:
def difficulty_scores(dataset: Dataset) -> np.ndarray:
    length = np.array([len(tok) for tok in dataset["tokens"]], dtype=float)
    length = (length - length.mean()) / length.std()

    dens = []
    for labels in dataset["ner_tags"]:
        non_o = sum(1 for l in labels if l != "O")
        dens.append(non_o / len(labels))
    dens = np.array(dens)
    dens = (dens - dens.mean()) / dens.std()

    ent_types, freq = [], Counter()
    for labels in dataset["ner_tags"]:
        types = [l[2:] for l in labels if l != "O"]
        ent_types.append(types[0] if types else "NONE")
    freq.update(ent_types)
    rarity = np.array([1 / freq[t] for t in ent_types])
    rarity = (rarity - rarity.mean()) / rarity.std()

    return length + dens + rarity


def reverse_curriculum_split(
    dataset: Dataset, pct_test: float = 0.20, pct_val: float = 0.10, seed: int = 42
) -> DatasetDict:
    scores = difficulty_scores(dataset)
    order = np.argsort(scores)  # easy→hard
    n_test = int(len(dataset) * pct_test)
    n_val = int(len(dataset) * pct_val)

    test_idx = order[-n_test:]  # hardest
    val_idx = order[-(n_test + n_val) : -n_test]
    train_idx = order[: -(n_test + n_val)]

    rng = np.random.RandomState(seed)
    rng.shuffle(train_idx)
    rng.shuffle(val_idx)
    rng.shuffle(test_idx)

    return DatasetDict(
        {
            "train": dataset.select(train_idx),
            "val": dataset.select(val_idx),
            "test": dataset.select(test_idx),
        }
    )

In [22]:
def heur_len_split(dataset: Dataset,
                   pct_test: float = 0.20,
                   pct_val : float = 0.10,
                   seed: int = 42) -> DatasetDict:
    """Testa só sentenças ≥ máx(len_train)."""
    sent_lens = np.array([len(t) for t in dataset["tokens"]])
    # separação inicial train/dev (randômica estratificando por tamanho grosso)
    idx_all   = np.arange(len(dataset))
    train_idx, temp_idx = train_test_split(idx_all,
                                           test_size=pct_test + pct_val,
                                           stratify=(sent_lens//5),  # bin len
                                           random_state=seed)
    # define longo-threshold como tamanho máx. do treino
    max_train_len = sent_lens[train_idx].max()
    # test = sentenças > threshold.  Caso falte/ sobre exemplos, ajusta.
    test_idx  = [i for i in temp_idx if sent_lens[i] > max_train_len]
    resto_idx = [i for i in temp_idx if i not in test_idx]
    # completa ou reduz para atingir pct_test
    need = int(pct_test*len(dataset)) - len(test_idx)
    if need > 0:
        test_idx.extend(resto_idx[:need])
        val_idx = resto_idx[need:]
    else:
        val_keep = int(pct_val*len(dataset))
        val_idx  = resto_idx[:val_keep]
        test_idx = test_idx[: int(pct_test*len(dataset))]
    return DatasetDict({
        "train": dataset.select(train_idx),
        "val"  : dataset.select(val_idx),
        "test" : dataset.select(test_idx),
    })

# ---------- 2) Heuristic Rare-Words ----------------------------------
def heur_rare_split(dataset: Dataset,
                    pct_test: float = 0.20,
                    pct_val : float = 0.10,
                    seed: int = 42) -> DatasetDict:
    """Testa frases que contenham palavras do quintil + raro."""
    # contagem de frequência de token
    freqs = Counter(w for toks in dataset["tokens"] for w in toks)
    # define rareza: 20 % mais raras
    thresh = np.quantile(list(freqs.values()), 0.20)
    rare_set = {w for w,c in freqs.items() if c <= thresh}
    is_rare = np.array([any(w in rare_set for w in toks)
                        for toks in dataset["tokens"]])
    rare_idx   = np.where(is_rare)[0]
    common_idx = np.where(~is_rare)[0]
    # garante proporções desejadas
    n_test = int(pct_test*len(dataset))
    n_val  = int(pct_val *len(dataset))
    rng = np.random.default_rng(seed)
    test_idx = rng.choice(rare_idx, size=min(len(rare_idx), n_test),
                          replace=False)
    resto_idx = [i for i in rare_idx if i not in test_idx] + list(common_idx)
    val_idx  = rng.choice(resto_idx, size=n_val, replace=False)
    train_idx = [i for i in resto_idx if i not in val_idx]
    return DatasetDict({
        "train": dataset.select(train_idx),
        "val"  : dataset.select(val_idx),
        "test" : dataset.select(test_idx),
    })

# ---------- 3) Standard (80-10-10) -----------------------------------
def std_split(dataset: Dataset,
              pct_test: float = 0.10,
              pct_val : float = 0.10,
              seed: int = 42) -> DatasetDict:
    """Split aleatório estratificado por comprimento (PTB-like)."""
    idx = np.arange(len(dataset))
    strat = (np.array([len(t) for t in dataset["tokens"]]) // 5)
    train_idx, temp_idx = train_test_split(idx, test_size=pct_test+pct_val,
                                          stratify=strat, random_state=seed)
    val_rel = pct_val / (pct_test+pct_val)
    val_idx, test_idx = train_test_split(temp_idx, test_size=1-val_rel,
                                         stratify=strat[temp_idx],
                                         random_state=seed)
    return DatasetDict({
        "train": dataset.select(train_idx),
        "val"  : dataset.select(val_idx),
        "test" : dataset.select(test_idx),
    })

# ---------- 4) Adversarial (approx. Wasserstein) ---------------------
def adversarial_split(dataset: Dataset,
                      pct_test: float = 0.20,
                      pct_val : float = 0.10,
                      k: int = 5,
                      seed: int = 42) -> DatasetDict:
    """
    Seleciona k frases 'mais distantes' recursivamente (BallTree + W₂)
    para compor o teste, lembrando Alg.-1 de Søgaard et al.【turn6file4】.
    """
    # SBERT embed (rápido na GPU / aceitável CPU)
    sbert = SentenceTransformer(
        "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
    )
    emb = sbert.encode([" ".join(toks) for toks in dataset["tokens"]],
                       batch_size=64, show_progress_bar=False)
    # BallTree para vizinhança eficiente
    tree = BallTree(emb, leaf_size=40)
    idx_pool = set(range(len(dataset)))
    test_idx = []
    rng = np.random.default_rng(seed)
    while len(test_idx) < int(pct_test*len(dataset)):
        # amostra candidata: ponto + longe do centro
        center = emb[list(idx_pool)].mean(0, keepdims=True)
        dists, _ = tree.query(center, k=len(idx_pool))
        farthest = list(idx_pool)[int(dists.argmax())]
        # pega-se farest e seus k-NN mais próximos  ⇒ aumenta diversidade
        nn = tree.query([emb[farthest]], k=k, return_distance=False)[0]
        for j in nn:
            if j in idx_pool and len(test_idx) < int(pct_test*len(dataset)):
                test_idx.append(j)
                idx_pool.remove(j)
    # retira val
    val_size = int(pct_val*len(dataset))
    val_idx  = rng.choice(list(idx_pool), size=val_size, replace=False)
    idx_pool -= set(val_idx)
    train_idx = list(idx_pool)
    return DatasetDict({
        "train": dataset.select(train_idx),
        "val"  : dataset.select(val_idx),
        "test" : dataset.select(test_idx),
    })

In [23]:
def heur_len_split(dataset: Dataset,
                   pct_test: float = 0.20,
                   pct_val : float = 0.10,
                   seed: int = 42,
                   bin_size: int = 10) -> DatasetDict:
    """
    Teste = sentenças mais longas que max(len(train)).
    Robusto a datasets pequenos: se estratificação falhar, usa split aleatório.
    """
    rng = np.random.default_rng(seed)
    idx_all   = np.arange(len(dataset))
    sent_lens = np.array([len(t) for t in dataset["tokens"]])

    # ------ 1) tenta split estratificado por baldes -------------------------
    strat = (sent_lens // bin_size)
    try:
        train_idx, temp_idx = train_test_split(
            idx_all,
            test_size=pct_test + pct_val,
            stratify=strat,
            random_state=seed,
        )
    except ValueError:                       # classes com 1 amostra
        train_idx, temp_idx = train_test_split(
            idx_all,
            test_size=pct_test + pct_val,
            shuffle=True,
            random_state=seed,
            stratify=None,
        )

    # ------ 2) escolhe test = len > max(train) ------------------------------
    max_train_len = sent_lens[train_idx].max()
    test_idx  = [i for i in temp_idx if sent_lens[i] > max_train_len]
    resto_idx = [i for i in temp_idx if i not in test_idx]

    # garante tamanhos exatos
    n_test_desired = int(pct_test * len(dataset))
    n_val_desired  = int(pct_val  * len(dataset))

    # completa teste se ficou pequeno
    if len(test_idx) < n_test_desired:
        extra = rng.choice(resto_idx,
                           size=n_test_desired - len(test_idx),
                           replace=False)
        test_idx.extend(extra)
        resto_idx = [i for i in resto_idx if i not in extra]

    # define validação
    val_idx  = rng.choice(resto_idx, size=n_val_desired, replace=False)
    train_idx = [i for i in idx_all if i not in test_idx and i not in val_idx]

    return DatasetDict({
        "train": dataset.select(train_idx),
        "val"  : dataset.select(val_idx),
        "test" : dataset.select(test_idx),
    })

In [24]:
def std_split(dataset,
              pct_test: float = 0.10,
              pct_val : float = 0.10,
              seed: int = 42,
              bin_size: int = 5) -> DatasetDict:
    """
    Split 80-10-10 robusto.
      • Tenta estratificar por comprimento // bin_size.
      • Se houver classes com <2 amostras, recua p/ split aleatório.
    """
    idx  = np.arange(len(dataset))
    bins = (np.array([len(t) for t in dataset["tokens"]]) // bin_size)

    try:
        train_idx, temp_idx = train_test_split(
            idx,
            test_size=pct_test + pct_val,
            stratify=bins,
            random_state=seed,
        )
    except ValueError:                       # classes muito pequenas
        train_idx, temp_idx = train_test_split(
            idx,
            test_size=pct_test + pct_val,
            shuffle=True,
            random_state=seed,
            stratify=None,
        )

    # fraciona temp em val / test mantendo proporção desejada
    val_share = pct_val / (pct_test + pct_val)
    try:
        val_idx, test_idx = train_test_split(
            temp_idx,
            test_size=1 - val_share,
            stratify=bins[temp_idx],
            random_state=seed,
        )
    except ValueError:
        val_idx, test_idx = train_test_split(
            temp_idx,
            test_size=1 - val_share,
            shuffle=True,
            random_state=seed,
            stratify=None,
        )

    return DatasetDict({
        "train": dataset.select(train_idx),
        "val"  : dataset.select(val_idx),
        "test" : dataset.select(test_idx),
    })

In [25]:
def adversarial_split(dataset: Dataset,
                      pct_test: float = 0.20,
                      pct_val : float = 0.10,
                      k: int = 5,
                      seed: int = 42) -> DatasetDict:
    """
    Versão robusta: nunca “trava” antes de atingir n_test.
    Seleciona blocos de k sentenças mais distantes do centro iterativamente.
    """
    # -------------------------------- embeds -------------------------------
    sbert = SentenceTransformer(
        "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
    )
    emb = sbert.encode([" ".join(t) for t in dataset["tokens"]],
                       batch_size=64, show_progress_bar=False)
    tree = BallTree(emb, leaf_size=40)

    n_test = int(pct_test * len(dataset))
    n_val  = int(pct_val  * len(dataset))

    idx_pool = set(range(len(dataset)))
    test_idx = []
    rng = np.random.default_rng(seed)

    print(f"Selecionando {n_test} sentenças para teste…")
    step = 0
    while len(test_idx) < n_test and idx_pool:
        if step % 5 == 0:
            print(f"  {len(test_idx)} selecionadas…")
        step += 1

        pool_list = list(idx_pool)
        center = emb[pool_list].mean(0, keepdims=True)

        # distância euclidiana ao centro global restante
        dists, _ = tree.query(center, k=len(pool_list))
        farthest_local_idx = int(dists.argmax())
        farthest_global_idx = pool_list[farthest_local_idx]

        # número efetivo de vizinhos
        k_eff = min(k, len(idx_pool))
        nn = set(tree.query([emb[farthest_global_idx]],
                            k=k_eff,
                            return_distance=False)[0])

        # adiciona vizinhos ainda não selecionados
        for j in nn:
            if j in idx_pool and len(test_idx) < n_test:
                test_idx.append(j)
                idx_pool.remove(j)

        # Se nada foi adicionado (pode acontecer quando sobram <k únicos)
        if farthest_global_idx not in test_idx:
            test_idx.append(farthest_global_idx)
            idx_pool.remove(farthest_global_idx)

    # ------------------------ validação e treino ---------------------------
    val_idx = rng.choice(list(idx_pool), size=n_val, replace=False)
    idx_pool -= set(val_idx)
    train_idx = list(idx_pool)

    return DatasetDict({
        "train": dataset.select(train_idx),
        "val"  : dataset.select(val_idx),
        "test" : dataset.select(test_idx),
    })

In [26]:
def _zscore_safe(x):
    x = np.asarray(x, dtype=float)
    std = x.std()
    if std == 0 or not np.isfinite(std):
        return np.zeros_like(x)
    return (x - x.mean()) / std


def difficulty_scores(dataset, id2label=None, o_label="O"):
    """
    dataset: HF Dataset com colunas 'tokens' e 'ner_tags_str' (ou 'ner_tags' + id2label).
    id2label: dict opcional para converter IDs -> string (se você só tiver 'ner_tags').
    """
    tokens_list = dataset["tokens"]
    # pega labels na forma de strings
    if "ner_tags_str" in dataset.column_names:
        labels_list = dataset["ner_tags_str"]
    else:
        assert (
            "ner_tags" in dataset.column_names and id2label is not None
        ), "Forneça id2label se tiver apenas ner_tags numéricos."
        labels_list = [[id2label[i] for i in seq] for seq in dataset["ner_tags"]]

    # 1) comprimento da sentença
    length = _zscore_safe([len(toks) for toks in tokens_list])

    # 2) densidade de entidades (proporção de tags != O)
    dens_raw = []
    for labels in labels_list:
        n = len(labels)
        non_o = sum(1 for l in labels if l != o_label)
        dens_raw.append(non_o / max(1, n))  # evita divisão por zero
    dens = _zscore_safe(dens_raw)

    # 3) raridade do tipo de entidade "principal" da sentença
    ent_types = []
    for labels in labels_list:
        # pega tipos removendo o prefixo BIO (B-/I-)
        types = [l.split("-", 1)[1] for l in labels if l != o_label and "-" in l]
        ent_types.append(types[0] if types else "NONE")
    freq = Counter(ent_types)
    rarity_raw = [1.0 / freq[t] for t in ent_types]
    rarity = _zscore_safe(rarity_raw)

    return length + dens + rarity


def reverse_curriculum_split(
    dataset, pct_test: float = 0.20, pct_val: float = 0.10, seed: int = 42, **kwargs
) -> DatasetDict:
    """
    kwargs são repassados para difficulty_scores (ex.: id2label=..., o_label="O").
    """
    scores = difficulty_scores(dataset, **kwargs)
    order = np.argsort(scores)  # easy -> hard

    n = len(dataset)
    n_test = int(round(n * pct_test))
    n_val = int(round(n * pct_val))
    n_test = min(max(n_test, 1), n - 2) if n >= 3 else max(n_test, 1)
    n_val = min(max(n_val, 1), n - 1 - n_test) if n - n_test >= 2 else max(n_val, 0)

    test_idx = order[-n_test:]
    val_idx = (
        order[-(n_test + n_val) : -n_test] if n_val > 0 else np.array([], dtype=int)
    )
    train_idx = order[: max(0, n - (n_test + n_val))]

    rng = np.random.RandomState(seed)
    rng.shuffle(train_idx)
    rng.shuffle(val_idx)
    rng.shuffle(test_idx)

    return DatasetDict(
        {
            "train": dataset.select(train_idx.tolist()),
            "val": (
                dataset.select(val_idx.tolist()) if len(val_idx) else dataset.select([])
            ),
            "test": dataset.select(test_idx.tolist()),
        }
    )

In [27]:
standard_split = std_split(lener_full)
print('std')
# random_splt = random_splits(lener_full)
# print('random')
heur_len = heur_len_split(lener_full)
print("heur_len")
heur_rare = heur_rare_split(lener_full)
print("heur_rare")
advers = adversarial_split(lener_full)
print("advs")
loc = loc_split(lener_full)
print("loc")
semantic = semantic_cluster_split(lener_full)
print("semantic")
reverse = reverse_curriculum_split(lener_full)
print("reverse")

std
heur_len
heur_rare
Selecionando 2079 sentenças para teste…
  0 selecionadas…
  25 selecionadas…
  50 selecionadas…
  74 selecionadas…
  95 selecionadas…
  118 selecionadas…
  142 selecionadas…
  165 selecionadas…
  187 selecionadas…
  209 selecionadas…
  229 selecionadas…
  251 selecionadas…
  275 selecionadas…
  298 selecionadas…
  320 selecionadas…
  345 selecionadas…
  370 selecionadas…
  392 selecionadas…
  408 selecionadas…
  422 selecionadas…
  446 selecionadas…
  466 selecionadas…
  486 selecionadas…
  499 selecionadas…
  517 selecionadas…
  536 selecionadas…
  551 selecionadas…
  572 selecionadas…
  594 selecionadas…
  608 selecionadas…
  622 selecionadas…
  639 selecionadas…
  656 selecionadas…
  669 selecionadas…
  690 selecionadas…
  712 selecionadas…
  734 selecionadas…
  752 selecionadas…
  766 selecionadas…
  782 selecionadas…
  802 selecionadas…
  816 selecionadas…
  840 selecionadas…
  859 selecionadas…
  871 selecionadas…
  894 selecionadas…
  914 selecionadas…
  9

100%|██████████| 10395/10395 [00:00<00:00, 709920.87it/s]


semantic
reverse


In [28]:
standard_split

DatasetDict({
    train: Dataset({
        features: ['id', 'tokens', 'ner_tags', 'ner_tags_str'],
        num_rows: 8316
    })
    val: Dataset({
        features: ['id', 'tokens', 'ner_tags', 'ner_tags_str'],
        num_rows: 1039
    })
    test: Dataset({
        features: ['id', 'tokens', 'ner_tags', 'ner_tags_str'],
        num_rows: 1040
    })
})

In [29]:
heur_len

DatasetDict({
    train: Dataset({
        features: ['id', 'tokens', 'ner_tags', 'ner_tags_str'],
        num_rows: 7277
    })
    val: Dataset({
        features: ['id', 'tokens', 'ner_tags', 'ner_tags_str'],
        num_rows: 1039
    })
    test: Dataset({
        features: ['id', 'tokens', 'ner_tags', 'ner_tags_str'],
        num_rows: 2079
    })
})

In [30]:
heur_rare

DatasetDict({
    train: Dataset({
        features: ['id', 'tokens', 'ner_tags', 'ner_tags_str'],
        num_rows: 7277
    })
    val: Dataset({
        features: ['id', 'tokens', 'ner_tags', 'ner_tags_str'],
        num_rows: 1039
    })
    test: Dataset({
        features: ['id', 'tokens', 'ner_tags', 'ner_tags_str'],
        num_rows: 2079
    })
})

In [31]:
from collections import Counter, defaultdict
from typing import Dict, List, Tuple
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_distances

In [32]:
def _label_names(ds_split):
    """Tenta obter nomes dos rótulos a partir do schema do HF Datasets."""
    feat = ds_split.features["ner_tags"]
    if hasattr(feat, "feature") and hasattr(feat.feature, "names"):
        return list(feat.feature.names)
    # fallback: cria nomes genéricos a partir dos valores observados
    vals = set()
    for ex in ds_split:
        vals.update(ex["ner_tags"])
    return list(vals)

def _flatten_labels(ds_split):
    for ex in ds_split:
        for y in ex["ner_tags"]:
            yield y

def _sent_texts(ds_split):
    # junta tokens com espaço para TF-IDF
    for ex in ds_split:
        yield " ".join(ex["tokens"])

def _token_stream(ds_split):
    for ex in ds_split:
        for t in ex["tokens"]:
            yield t

def _entity_counts(ds_split, id2name):
    # conta rótulos (token-level) e também sequência de entidades (simples: conta B-*)
    c_tok = Counter()
    for ex in ds_split:
        for y in ex["ner_tags"]:
             c_tok[y] += 1
    return c_tok

def _lengths(ds_split):
    return np.array([len(ex["tokens"]) for ex in ds_split], dtype=int)

def _oov_rate(train_vocab: set, ds_split) -> float:
    tot = 0
    oov = 0
    for ex in ds_split:
        for t in ex["tokens"]:
            tot += 1
            if t not in train_vocab:
                oov += 1
    return oov / max(tot, 1)

def _tfidf_centroid(texts: List[str], vectorizer: TfidfVectorizer=None):
    # retorna (centro, vectorizer)
    fit_new = vectorizer is None
    if fit_new:
        vectorizer = TfidfVectorizer(min_df=2, max_df=0.98, ngram_range=(1,2))
        X = vectorizer.fit_transform(texts)
    else:
        X = vectorizer.transform(texts)
    # média das linhas (TF-IDF já vem L2-normalizado por linha)
    centroid = X.mean(axis=0)                       # <- vira np.matrix
    centroid = np.asarray(centroid).ravel()         # -> ndarray (D,)
    # (opcional) normalizar o centróide para norma 1, por simetria do cosseno
    denom = norm(centroid)
    if denom > 0:
        centroid = centroid / denom
    centroid = centroid.reshape(1, -1)              # -> (1, D) para pairwise
    return centroid, vectorizer

def _cos(a, b):
    """
    Distância do cosseno com entrada tolerante (matrix/array/1D/2D).
    Retorna float.
    """
    a = np.asarray(a)
    b = np.asarray(b)
    if a.ndim == 1:
        a = a.reshape(1, -1)
    if b.ndim == 1:
        b = b.reshape(1, -1)
    return float(cosine_distances(a, b)[0, 0])

In [33]:
def analyze_instances_for_insights(ds_dict, top_k_tokens=20):
    id2name = _label_names(ds_dict["train"])

    # comprimentos
    lens = {p: np.array([len(ex["tokens"]) for ex in ds_dict[p]], int)
            for p in ["train","val","test"]}
    q = {p: tuple(map(int, np.quantile(lens[p], [0.1, 0.5, 0.9, 0.99])))
         for p in lens}

    # OOV: val/test vs vocabulário do treino
    train_vocab = set(_token_stream(ds_dict["train"]))
    oov_val  = _oov_rate(train_vocab, ds_dict["val"])
    oov_test = _oov_rate(train_vocab, ds_dict["test"])

    # rótulos raros (no conjunto completo)
    total_counts = Counter()
    for p in ["train","val","test"]:
        total_counts.update(_entity_counts(ds_dict[p], id2name))
    rares = sorted(total_counts.items(), key=lambda kv: kv[1])[:5]

    # tokens mais característicos (teste vs treino) por log-odds simples
    def _log_odds(alpha=0.01):
        c_tr = Counter(_token_stream(ds_dict["train"]))
        c_te = Counter(_token_stream(ds_dict["test"]))
        V = set(c_tr) | set(c_te)
        n_tr, n_te = sum(c_tr.values()), sum(c_te.values())
        scores = [(w, np.log((c_te[w]+alpha)/(n_te+alpha*len(V))) -
                      np.log((c_tr[w]+alpha)/(n_tr+alpha*len(V)))) for w in V]
        scores.sort(key=lambda x: x[1], reverse=True)
        return [w for w,_ in scores[:top_k_tokens]], [w for w,_ in scores[-top_k_tokens:]]

    top_test, top_train = _log_odds()

    return {
        "length_quantiles": q,  # p10, p50, p90, p99 por split
        "oov_rate_val_vs_train": oov_val,
        "oov_rate_test_vs_train": oov_test,
        "5_rare_labels_overall_(label,count)": rares,
        "tokens_more_characteristic_in_test": top_test,
        "tokens_more_characteristic_in_train": top_train,
    }

In [34]:
def save_split_insights_to_csv(split_name: str, ins: dict, out_root: str = "insights_out") -> None:
    """
    Salva cada parte de `ins` (dict) em um CSV distinto, dentro de insights_out/<split_name>/.
    Estruturas:
      - length_quantiles.csv: linhas longas (part, quantile, value)
      - oov_rates.csv       : (metric, value)
      - rare_labels.csv     : (label, count) [top-5]
      - tokens_test.csv     : (rank, token)
      - tokens_train.csv    : (rank, token)
    """
    out_dir = os.path.join(out_root, split_name)
    os.makedirs(out_dir, exist_ok=True)

    # 1) Quantis de comprimento (formato longo)
    # ins["length_quantiles"] = { 'train': (p10,p50,p90,p99), 'val': ..., 'test': ... }
    rows = []
    for part, quads in ins["length_quantiles"].items():
        p10, p50, p90, p99 = quads
        rows.extend([
            {"part": part, "quantile": "p10", "value": int(p10)},
            {"part": part, "quantile": "p50", "value": int(p50)},
            {"part": part, "quantile": "p90", "value": int(p90)},
            {"part": part, "quantile": "p99", "value": int(p99)},
        ])
    pd.DataFrame(rows).to_csv(os.path.join(out_dir, "length_quantiles.csv"), index=False)

    # 2) OOV val/test vs train
    pd.DataFrame(
        [
            {"metric": "oov_rate_val_vs_train",  "value": float(ins["oov_rate_val_vs_train"])},
            {"metric": "oov_rate_test_vs_train", "value": float(ins["oov_rate_test_vs_train"])},
        ]
    ).to_csv(os.path.join(out_dir, "oov_rates.csv"), index=False)

    # 3) Rótulos raros (top-5)
    # chave tem um nome longo; localiza pela substring "rare_labels"
    rare_key = next(k for k in ins.keys() if "rare_labels" in k)
    # ins[rare_key] = [(label, count), ...]
    pd.DataFrame(ins[rare_key], columns=["label", "count"]).to_csv(
        os.path.join(out_dir, "rare_labels.csv"), index=False
    )

    # 4) Tokens mais característicos (teste vs treino)
    pd.DataFrame(
        {
            "rank": list(range(1, len(ins["tokens_more_characteristic_in_test"]) + 1)),
            "token": ins["tokens_more_characteristic_in_test"],
        }
    ).to_csv(os.path.join(out_dir, "tokens_test.csv"), index=False)

    pd.DataFrame(
        {
            "rank": list(range(1, len(ins["tokens_more_characteristic_in_train"]) + 1)),
            "token": ins["tokens_more_characteristic_in_train"],
        }
    ).to_csv(os.path.join(out_dir, "tokens_train.csv"), index=False)

In [35]:
splits = {
    "standard": standard_split,
    "heur_len": heur_len,
    "heur_rare": heur_rare,
    "adversarial": advers,
    "loc": loc,
    "semantic": semantic,
    "reverse": reverse,
}

# 1) Insights por split
insights = {}
for name, ds_dict in splits.items():
    ins = analyze_instances_for_insights(ds_dict, top_k_tokens=20)
    insights[name] = ins
    print(f"\n[{name}]")
    print(ins)
    save_split_insights_to_csv(name, ins, out_root="insights_out")


[standard]
{'length_quantiles': {'train': (3, 21, 68, 151), 'val': (2, 22, 69, 161), 'test': (2, 21, 71, 163)}, 'oov_rate_val_vs_train': 0.036283517719857494, 'oov_rate_test_vs_train': 0.03636136448831688, '5_rare_labels_overall_(label,count)': [(7, 767), (8, 1026), (6, 1386), (11, 1496), (5, 1760)], 'tokens_more_characteristic_in_test': ['CORRÉU', 'Caixa', '788/94', 'claros', 'Econômica', 'Extra', "'IN", 'Goiás', 'CCJ', 'BERTON', 'empréstimo', 'explicar', 'destacam', 'conta-corrente', '0394', 'sucintamente', 'Norte', 'assentar', 'franquia', 'estratégico'], 'tokens_more_characteristic_in_train': ['Primeira', 'REspe', 'condão', 'encontra-se', 'dar', 'depoente', 'denúncia', '7º', 'mera', 'Dessa', 'juntada', 'contrariedade', 'início', '1999', 'comprovação', 'alterações', 'documentação', 'tocante', 'DIRETA', 'obra']}

[heur_len]
{'length_quantiles': {'train': (3, 22, 68, 155), 'val': (2, 20, 68, 128), 'test': (2, 22, 69, 161)}, 'oov_rate_val_vs_train': 0.03620238922532182, 'oov_rate_test_

In [36]:
def class_distribution_table(ds_dict) -> Tuple[pd.DataFrame, Dict[str, np.ndarray]]:
    id2name_train = _label_names(ds_dict["train"])

    # 1) Contagens por part e união de labels observadas em todo o DatasetDict
    counts_by_part = {}
    labels_union = set(id2name_train)
    for part in ["train", "val", "test"]:
        c = _entity_counts(ds_dict[part], id2name_train)  # mantém sua assinatura atual
        counts_by_part[part] = c
        labels_union |= set(c.keys())

    # 2) Ordem estável: primeiro as labels do treino, depois as novas (ordenadas)
    new_labels = sorted(labels_union - set(id2name_train))
    labels_all = list(id2name_train) + new_labels

    name2id = {n: i for i, n in enumerate(labels_all)}
    n_labels = len(labels_all)

    # 3) Monta DF longo + vetores normalizados por split
    rows = []
    label_vecs = {}
    for part in ["train", "val", "test"]:
        c = counts_by_part[part]
        total = int(sum(c.values()))
        vec = np.zeros((1, n_labels), dtype=float)

        # preenche vetor e linhas do DF na ordem labels_all (inclui zero para rótulos ausentes)
        for lab in labels_all:
            cnt = int(c.get(lab, 0))
            vec[0, name2id[lab]] = cnt
            rows.append((part, lab, cnt, cnt / max(total, 1)))

        # normaliza pelo total do part (evita div by zero)
        label_vecs[part] = vec / max(total, 1)

    class_df = pd.DataFrame(rows, columns=["split", "label", "count", "prop"])
    return class_df, label_vecs

In [37]:
def save_class_distribution_to_csv(split_name: str,
                                   class_df: pd.DataFrame,
                                   label_vecs: dict,
                                   out_root: str = "class_dist_out") -> None:
    # cria pasta por split
    out_dir = os.path.join(out_root, split_name)
    os.makedirs(out_dir, exist_ok=True)

    # 1) DF longo original
    class_df.to_csv(os.path.join(out_dir, "class_distribution_long.csv"), index=False)

    # 2) Tabelas dinâmicas (counts e props)
    counts = class_df.pivot(index="label", columns="split", values="count").fillna(0).astype(int)
    props  = class_df.pivot(index="label", columns="split", values="prop").fillna(0.0)

    counts.to_csv(os.path.join(out_dir, "counts_pivot.csv"))
    props.to_csv(os.path.join(out_dir, "props_pivot.csv"))

    # 3) label_vecs: normalizado por split (cada linha = um split; colunas = rótulos)
    # label_vecs[part] = array shape (1, n_labels); precisamos do nome das colunas na ordem correta
    # recupera a ordem das labels do DF original
    labels_order = (
        class_df[["label"]]
        .drop_duplicates()["label"]
        .tolist()
    )
    rows = []
    for part, vec in label_vecs.items():
        row = {"split": part}
        for i, lab in enumerate(labels_order):
            row[lab] = float(vec[0, i]) if vec.ndim == 2 else float(vec[i])
        rows.append(row)
    lv_df = pd.DataFrame(rows).set_index("split")
    lv_df = lv_df[labels_order]  # garante ordem das colunas
    lv_df.to_csv(os.path.join(out_dir, "label_vecs.csv"))

    # 4) também salva um CSV por part (opcional, útil para comparar rápido)
    for part in label_vecs:
        pd.DataFrame([lv_df.loc[part]], index=[part]).to_csv(
            os.path.join(out_dir, f"label_vecs_{part}.csv")
        )

In [38]:
# =========================================================
# 2) Análise quantitativa das classes (train/val/test)
# =========================================================
class_distribution = {}
for name, ds_dict in splits.items():
    df_long, vecs = class_distribution_table(ds_dict)
    class_distribution[name] = df_long
    print(f"\n[{name}]")
    print(df_long.head())  # evita imprimir tudo
    save_class_distribution_to_csv(name, df_long, vecs, out_root="class_dist_out")


[standard]
   split          label  count  prop
0  train              O      0   0.0
1  train  B-ORGANIZACAO      0   0.0
2  train  I-ORGANIZACAO      0   0.0
3  train       B-PESSOA      0   0.0
4  train       I-PESSOA      0   0.0

[heur_len]
   split          label  count  prop
0  train              O      0   0.0
1  train  B-ORGANIZACAO      0   0.0
2  train  I-ORGANIZACAO      0   0.0
3  train       B-PESSOA      0   0.0
4  train       I-PESSOA      0   0.0

[heur_rare]
   split          label  count  prop
0  train              O      0   0.0
1  train  B-ORGANIZACAO      0   0.0
2  train  I-ORGANIZACAO      0   0.0
3  train       B-PESSOA      0   0.0
4  train       I-PESSOA      0   0.0

[adversarial]
   split          label  count  prop
0  train              O      0   0.0
1  train  B-ORGANIZACAO      0   0.0
2  train  I-ORGANIZACAO      0   0.0
3  train       B-PESSOA      0   0.0
4  train       I-PESSOA      0   0.0

[loc]
   split          label  count  prop
0  train        

In [39]:
def cosine_within_split(ds_dict) -> pd.DataFrame:
    # palavras (TF-IDF) — um vocabulário por split já é suficiente aqui
    tr_texts = list(_sent_texts(ds_dict["train"]))
    va_texts = list(_sent_texts(ds_dict["val"]))
    te_texts = list(_sent_texts(ds_dict["test"]))
    cen_tr, vec = _tfidf_centroid(tr_texts, None)
    cen_va, _   = _tfidf_centroid(va_texts, vec)
    cen_te, _   = _tfidf_centroid(te_texts, vec)

    # rótulos
    _, label_vecs = class_distribution_table(ds_dict)

    rows = [
        ["labels","train","val",  _cos(label_vecs["train"], label_vecs["val"])],
        ["labels","train","test", _cos(label_vecs["train"], label_vecs["test"])],
        ["labels","val","test",   _cos(label_vecs["val"],   label_vecs["test"])],
        ["words","train","val",   _cos(cen_tr, cen_va)],
        ["words","train","test",  _cos(cen_tr, cen_te)],
        ["words","val","test",    _cos(cen_va, cen_te)],
    ]
    return pd.DataFrame(rows, columns=["space","a","b","cosine_distance"])

In [40]:

def save_cosine_to_csv(split_name: str,
                       cos_df: pd.DataFrame,
                       out_root: str = "cosine_out") -> None:
    # cria pasta do split
    out_dir = os.path.join(out_root, split_name)
    os.makedirs(out_dir, exist_ok=True)

    # 1) DF completo
    cos_df.to_csv(os.path.join(out_dir, "cosine_all.csv"), index=False)

    # 2) Subconjuntos por espaço
    df_labels = cos_df[cos_df["space"] == "labels"].reset_index(drop=True)
    df_words  = cos_df[cos_df["space"] == "words"].reset_index(drop=True)

    df_labels.to_csv(os.path.join(out_dir, "cosine_labels.csv"), index=False)
    df_words.to_csv(os.path.join(out_dir, "cosine_words.csv"), index=False)

    # 3) Matrizes em formato largo (úteis para comparação rápida)
    # cada célula representa o cos(a,b)
    def _to_matrix(df_space: pd.DataFrame) -> pd.DataFrame:
        # cria coluna "pair" = "a_b" para pivot
        tmp = df_space.assign(pair=df_space["a"] + "_" + df_space["b"])
        mat = tmp.pivot_table(index="space", columns="pair",
                              values="cosine_distance", aggfunc="first")
        # A linha é o próprio "space" (labels/words); tiramos o índice para salvar “flat”
        mat.reset_index(drop=True, inplace=True)
        return mat

    _to_matrix(df_labels).to_csv(os.path.join(out_dir, "matrix_labels.csv"), index=False)
    _to_matrix(df_words).to_csv(os.path.join(out_dir, "matrix_words.csv"), index=False)


In [41]:
# =========================================================
# 3) Distância do cosseno entre train/val/test (por split)
#    (labels e palavras/TF-IDF)
# =========================================================
coisine = {}
all_rows = []  # para o resumo de todos os splits
for name, ds_dict in splits.items():
    df = cosine_within_split(ds_dict)
    coisine[name] = df
    print(f"\n[{name}]")
    print(df)
    save_cosine_to_csv(name, df, out_root="cosine_out")

    # anexa nome do split para o resumo agregado
    df_with_split = df.copy()
    df_with_split.insert(0, "split", name)
    all_rows.append(df_with_split)


[standard]
    space      a     b  cosine_distance
0  labels  train   val         0.000058
1  labels  train  test         0.000012
2  labels    val  test         0.000053
3   words  train   val         0.049936
4   words  train  test         0.044608
5   words    val  test         0.081449

[heur_len]
    space      a     b  cosine_distance
0  labels  train   val         0.000018
1  labels  train  test         0.000024
2  labels    val  test         0.000014
3   words  train   val         0.049502
4   words  train  test         0.027556
5   words    val  test         0.063345

[heur_rare]
    space      a     b  cosine_distance
0  labels  train   val         0.000041
1  labels  train  test         0.000047
2  labels    val  test         0.000016
3   words  train   val         0.044673
4   words  train  test         0.062345
5   words    val  test         0.094674

[adversarial]
    space      a     b  cosine_distance
0  labels  train   val         0.000030
1  labels  train  test      

In [42]:
summary = pd.concat(all_rows, ignore_index=True)
os.makedirs("cosine_out", exist_ok=True)
summary.to_csv(os.path.join("cosine_out", "cosine_summary_all_splits.csv"), index=False)

In [43]:
# =========================================================
def cosine_between_splits(splits: Dict[str, "DatasetDict"]) -> Dict[str, pd.DataFrame]:
    names = list(splits.keys())

    # --- (A) preparar centróides TF-IDF com VOCABULÁRIO COMPARTILHADO
    # fit no conjunto de todos os trains para comparabilidade direta
    all_train_texts = []
    for name in names:
        all_train_texts.extend(_sent_texts(splits[name]["train"]))
    shared_vec = TfidfVectorizer(min_df=2, max_df=0.98, ngram_range=(1,2)).fit(list(all_train_texts))

    word_centers_train = {}
    word_centers_test  = {}
    for name in names:
        cen_tr, _ = _tfidf_centroid(list(_sent_texts(splits[name]["train"])), shared_vec)
        cen_te, _ = _tfidf_centroid(list(_sent_texts(splits[name]["test"])),  shared_vec)
        word_centers_train[name] = cen_tr
        word_centers_test[name]  = cen_te

    # --- (B) preparar vetores de rótulos normalizados
    label_vecs_train = {}
    label_vecs_test  = {}
    for name in names:
        _, lv = class_distribution_table(splits[name])
        label_vecs_train[name] = lv["train"]
        label_vecs_test[name]  = lv["test"]

    # --- (C) montar matrizes de distâncias (palavras e rótulos, train/test)
    def _pairwise(names, get_vec):
        M = np.zeros((len(names), len(names)))
        for i,a in enumerate(names):
            for j,b in enumerate(names):
                M[i,j] = _cos(get_vec(a), get_vec(b))
        return pd.DataFrame(M, index=names, columns=names)

    cos_train_words = _pairwise(names, lambda n: word_centers_train[n])
    cos_test_words  = _pairwise(names, lambda n: word_centers_test[n])
    cos_train_lbls  = _pairwise(names, lambda n: label_vecs_train[n])
    cos_test_lbls   = _pairwise(names, lambda n: label_vecs_test[n])

    return {
        "cos_train_words": cos_train_words,
        "cos_test_words":  cos_test_words,
        "cos_train_labels": cos_train_lbls,
        "cos_test_labels":  cos_test_lbls,
    }

In [44]:
between = cosine_between_splits(splits)

In [45]:
out_root = "cosine_between_out"
os.makedirs(out_root, exist_ok=True)

# 1) Salva as 4 matrizes completas
file_map = {
    "cos_train_words":  "cos_train_words.csv",
    "cos_test_words":   "cos_test_words.csv",
    "cos_train_labels": "cos_train_labels.csv",
    "cos_test_labels":  "cos_test_labels.csv",
}
for key, fname in file_map.items():
    between[key].to_csv(os.path.join(out_root, fname))


In [46]:
between["cos_train_words"]

,standard,heur_len,heur_rare,adversarial,loc,semantic,reverse
standard,0.000000,0.001205,0.007340,0.012499,0.012980,0.021658,0.037943
heur_len,0.001205,0.000000,0.008449,0.014051,0.014536,0.022559,0.037991
heur_rare,0.007340,0.008449,0.000000,0.017036,0.021269,0.034141,0.037312
adversarial,0.012499,0.014051,0.017036,0.000000,0.022208,0.032819,0.053801
loc,0.012980,0.014536,0.021269,0.022208,0.000000,0.019395,0.067893
semantic,0.021658,0.022559,0.034141,0.032819,0.019395,0.000000,0.062053
reverse,0.037943,0.037991,0.037312,0.053801,0.067893,0.062053,0.000000


In [47]:
between["cos_test_words"]

,standard,heur_len,heur_rare,adversarial,loc,semantic,reverse
standard,0.000000,0.051282,0.107468,0.154054,0.370094,0.372424,0.149479
heur_len,0.051282,0.000000,0.076178,0.131830,0.373341,0.366932,0.114444
heur_rare,0.107468,0.076178,0.000000,0.159752,0.441662,0.487600,0.122408
adversarial,0.154054,0.131830,0.159752,0.000000,0.447929,0.399322,0.220916
loc,0.370094,0.373341,0.441662,0.447929,0.000000,0.336022,0.514349
semantic,0.372424,0.366932,0.487600,0.399322,0.336022,0.000000,0.425769
reverse,0.149479,0.114444,0.122408,0.220916,0.514349,0.425769,0.000000


In [48]:
between["cos_train_labels"]

,standard,heur_len,heur_rare,adversarial,loc,semantic,reverse
standard,0.000000e+00,7.667462e-07,0.000007,0.000006,0.000022,0.000013,0.001073
heur_len,7.667462e-07,0.000000e+00,0.000012,0.000003,0.000028,0.000016,0.001032
heur_rare,7.165365e-06,1.206797e-05,0.000000,0.000021,0.000007,0.000007,0.001229
adversarial,5.716683e-06,3.308482e-06,0.000021,0.000000,0.000034,0.000021,0.001013
loc,2.153915e-05,2.817802e-05,0.000007,0.000034,0.000000,0.000006,0.001388
semantic,1.280038e-05,1.608552e-05,0.000007,0.000021,0.000006,0.000000,0.001288
reverse,1.073150e-03,1.031715e-03,0.001229,0.001013,0.001388,0.001288,0.000000


In [49]:
between["cos_test_labels"]

,standard,heur_len,heur_rare,adversarial,loc,semantic,reverse
standard,0.000000,0.000021,0.000013,0.000141,0.001302,0.000290,0.001976
heur_len,0.000021,0.000000,0.000050,0.000080,0.001621,0.000337,0.001663
heur_rare,0.000013,0.000050,0.000000,0.000183,0.001121,0.000247,0.002235
adversarial,0.000141,0.000080,0.000183,0.000000,0.001987,0.000636,0.001524
loc,0.001302,0.001621,0.001121,0.001987,0.000000,0.001303,0.006387
semantic,0.000290,0.000337,0.000247,0.000636,0.001303,0.000000,0.002711
reverse,0.001976,0.001663,0.002235,0.001524,0.006387,0.002711,0.000000
